In [0]:
%run /Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/resources/utils

In [0]:
import pytest
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType
from datetime import date

configs = parse_config_from_json(
    "/Workspace/Users/ashutoshp2@icloud.com/ecommerce_data_pipeline/config/ingestion_configs.json"
)

In [0]:
# Test transformation configs have required keys 

def test_config_has_struct_schema():
    for module in ["Products", "Customers", "Orders"]:
        assert "struct_schema" in configs[module], f"{module} missing 'struct_schema'"

def test_config_has_defaults():
    for module in ["Products", "Customers", "Orders"]:
        assert "defaults" in configs[module], f"{module} missing 'defaults'"

def test_config_has_module_key():
    for module in ["Products", "Customers", "Orders"]:
        assert "module" in configs[module], f"{module} missing 'module'"

In [0]:
# Test remove_nulls_for_key drops rows with null keys

def test_remove_nulls_single_key():
    df = spark.createDataFrame(
        [("P1", "Chair"), (None, "Desk"), ("P3", "Lamp")],
        ["product_id", "product_name"]
    )
    result = remove_nulls_for_key(df, "product_id")
    assert result.count() == 2
    ids = [r["product_id"] for r in result.collect()]
    assert None not in ids

def test_remove_nulls_composite_key():
    df = spark.createDataFrame(
        [("O1", "C1", "P1"), (None, "C2", "P2"), ("O3", None, "P3")],
        ["order_id", "customer_id", "product_id"]
    )
    result = remove_nulls_for_key(df, ["order_id", "customer_id", "product_id"])
    assert result.count() == 1

In [0]:
# Test fill_defaults fills nulls with configured values

def test_fill_defaults_products():
    schema = StructType([
        StructField("product_id", StringType()), StructField("product_name", StringType()),
        StructField("category", StringType()), StructField("sub_category", StringType()),
        StructField("state", StringType()), StructField("price_per_product", DoubleType())
    ])
    df = spark.createDataFrame([("P1", None, None, None, None, None)], schema)
    defaults = configs["Products"]["defaults"]
    result = fill_defaults(df, defaults).collect()[0]
    assert result["product_name"] == defaults["product_name"]
    assert result["category"] == defaults["category"]

def test_fill_defaults_customers():
    schema = StructType([
        StructField("customer_id", StringType()), StructField("customer_name", StringType()),
        StructField("email", StringType()), StructField("phone", StringType()),
        StructField("address", StringType()), StructField("segment", StringType()),
        StructField("country", StringType()), StructField("city", StringType()),
        StructField("state", StringType()), StructField("postal_code", StringType()),
        StructField("region", StringType())
    ])
    df = spark.createDataFrame([("C1", None, None, None, None, None, None, None, None, None, None)], schema)
    defaults = configs["Customers"]["defaults"]
    result = fill_defaults(df, defaults).collect()[0]
    assert result["customer_name"] == defaults["customer_name"]
    assert result["email"] == defaults["email"]
    assert result["region"] == defaults["region"]

def test_fill_defaults_orders():
    schema = StructType([
        StructField("row_id", IntegerType()), StructField("order_id", StringType()),
        StructField("order_date", StringType()), StructField("ship_date", StringType()),
        StructField("ship_mode", StringType()), StructField("customer_id", StringType()),
        StructField("product_id", StringType()), StructField("quantity", IntegerType()),
        StructField("price", DoubleType()), StructField("discount", DoubleType()),
        StructField("profit", DoubleType())
    ])
    df = spark.createDataFrame([(1, "O1", "1/1/2024", "5/1/2024", None, "C1", "P1", None, None, None, None)], schema)
    defaults = configs["Orders"]["defaults"]
    result = fill_defaults(df, defaults).collect()[0]
    assert result["ship_mode"] == defaults["ship_mode"]

def test_fill_defaults_preserves_existing():
    df = spark.createDataFrame(
        [("P1", "Chair", "Furniture", "Seats", "CA", 50.0)],
        ["product_id", "product_name", "category", "sub_category", "state", "price_per_product"]
    )
    defaults = configs["Products"]["defaults"]
    result = fill_defaults(df, defaults).collect()[0]
    assert result["product_name"] == "Chair"
    assert result["category"] == "Furniture"

In [0]:
# Test parse_struct_schema returns valid StructType

def test_parse_struct_schema_products():
    schema = parse_struct_schema(configs["Products"]["struct_schema"])
    assert isinstance(schema, StructType)
    assert schema.names == ["product_id", "product_name", "category", "sub_category", "state", "price_per_product"]

def test_parse_struct_schema_customers():
    schema = parse_struct_schema(configs["Customers"]["struct_schema"])
    assert isinstance(schema, StructType)
    assert schema.names == ["customer_id", "customer_name", "email", "phone", "address",
                            "segment", "country", "city", "state", "postal_code", "region"]

def test_parse_struct_schema_orders():
    schema = parse_struct_schema(configs["Orders"]["struct_schema"])
    assert isinstance(schema, StructType)
    assert "order_id" in schema.names
    assert "order_date" in schema.names

In [0]:
# Test: apply_schema selects only the schema columns

def test_apply_schema_drops_extra_columns():
    df = spark.createDataFrame(
        [("P1", "Chair", "Furniture", "Seats", "CA", 50.0, "extra_val")],
        ["product_id", "product_name", "category", "sub_category", "state", "price_per_product", "junk"]
    )
    schema = parse_struct_schema(configs["Products"]["struct_schema"])
    result = apply_schema(df, schema)
    assert "junk" not in result.columns
    assert result.columns == ["product_id", "product_name", "category", "sub_category", "state", "price_per_product"]

In [0]:
# Test parse_dates converts strings to date type

def test_parse_dates_with_format():
    df = spark.createDataFrame(
        [("15/3/2024", "20/3/2024")],
        ["order_date", "ship_date"]
    )
    result = parse_dates(df, {"order_date": "d/M/yyyy", "ship_date": "d/M/yyyy"}).collect()[0]
    assert result["order_date"] == date(2024, 3, 15)
    assert result["ship_date"] == date(2024, 3, 20)

def test_parse_dates_null_stays_null():
    df = spark.createDataFrame(
        [(None,)],
        schema=StructType([StructField("order_date", StringType())])
    )
    result = parse_dates(df, {"order_date": "d/M/yyyy"}).collect()[0]
    assert result["order_date"] is None

In [0]:
# Test deduplication removes duplicate rows

def test_drop_duplicates():
    df = spark.createDataFrame(
        [("P1", "Chair"), ("P1", "Chair"), ("P2", "Desk")],
        ["product_id", "product_name"]
    )
    result = df.dropDuplicates()
    assert result.count() == 2

In [0]:
# Test surrogate key generation adds a new column

def test_surrogate_key_added():
    df = spark.createDataFrame(
        [("P1", "Chair"), ("P2", "Desk")],
        ["product_id", "product_name"]
    )
    result = df.withColumn("product_key", F.monotonically_increasing_id())
    assert "product_key" in result.columns
    keys = [r["product_key"] for r in result.collect()]
    assert len(set(keys)) == 2  # keys are unique

In [0]:
# Test end-to-end dim transform for products

def test_dim_transform_products_end_to_end():
    df = spark.createDataFrame(
        [("P1", "Chair", "Furniture", "Seats", "CA", 50.0),
         ("P1", "Chair", "Furniture", "Seats", "CA", 50.0),
         (None, "Ghost", "Unknown", "None", "XX", 0.0),
         ("P3", None, None, None, None, None)],
        ["product_id", "product_name", "category", "sub_category", "state", "price_per_product"]
    )
    schema = parse_struct_schema(configs["Products"]["struct_schema"])
    defaults = configs["Products"]["defaults"]

    result = remove_nulls_for_key(df, "product_id")
    result = result.dropDuplicates()
    result = fill_defaults(result, defaults)
    result = apply_schema(result, schema)
    result = result.withColumn("product_key", F.monotonically_increasing_id())

    assert result.count() == 2
    assert "product_key" in result.columns
    p3 = result.filter(F.col("product_id") == "P3").collect()[0]
    assert p3["product_name"] == "unknown"


# Test: end-to-end dim transform for customers

def test_dim_transform_customers_end_to_end():
    df = spark.createDataFrame(
        [("C1", "Alice", "a@x.com", "111", "1 St", "Corp", "US", "NY", "NY", "10001", "East"),
         ("C1", "Alice", "a@x.com", "111", "1 St", "Corp", "US", "NY", "NY", "10001", "East"),
         (None, "Ghost", "g@x.com", "000", "0 St", "None", "US", "XX", "XX", "00000", "None"),
         ("C3", None, None, None, None, None, None, None, None, None, None)],
        ["customer_id", "customer_name", "email", "phone", "address",
         "segment", "country", "city", "state", "postal_code", "region"]
    )
    schema = parse_struct_schema(configs["Customers"]["struct_schema"])
    defaults = configs["Customers"]["defaults"]

    result = remove_nulls_for_key(df, "customer_id")
    result = result.dropDuplicates()
    result = fill_defaults(result, defaults)
    result = apply_schema(result, schema)
    result = result.withColumn("customer_key", F.monotonically_increasing_id())

    assert result.count() == 2 
    assert "customer_key" in result.columns
    c3 = result.filter(F.col("customer_id") == "C3").collect()[0]
    assert c3["customer_name"] == "unknown"
    assert c3["email"] == "unknown"
    assert c3["region"] == "unknown"
    
    c1 = result.filter(F.col("customer_id") == "C1").collect()[0]
    assert c1["customer_name"] == "Alice"
    assert c1["email"] == "a@x.com"

In [0]:
# Test end-to-end fact transform for orders

def test_fact_transform_orders_end_to_end():
    df = spark.createDataFrame(
        [(1, "O1", "15/3/2024", "20/3/2024", "Standard", "C1", "P1", 2, 100.0, 0.1, 10.0),
         (1, "O1", "15/3/2024", "20/3/2024", "Standard", "C1", "P1", 2, 100.0, 0.1, 10.0),
         (2, "O2", "1/1/2024", "5/1/2024", None, "C2", "P2", 1, 50.0, 0.0, 5.0),
         (3, None, "1/1/2024", "5/1/2024", "Express", "C3", "P3", 1, 30.0, 0.0, 3.0),
         (4, "O4", "10/6/2024", "15/6/2024", "Express", None, "P4", 3, 200.0, 0.2, 20.0)],
        ["row_id", "order_id", "order_date", "ship_date", "ship_mode",
         "customer_id", "product_id", "quantity", "price", "discount", "profit"]
    )
    schema = parse_struct_schema(configs["Orders"]["struct_schema"])
    defaults = configs["Orders"]["defaults"]
    date_cols = {"order_date": "d/M/yyyy", "ship_date": "d/M/yyyy"}

    result = remove_nulls_for_key(df, ["order_id", "customer_id", "product_id"])
    result = result.dropDuplicates()
    result = fill_defaults(result, defaults)
    result = parse_dates(result, date_cols)
    result = apply_schema(result, schema)
    result = result.withColumn("order_key", F.monotonically_increasing_id())

    assert result.count() == 2  # null keys removed, duplicate removed
    assert "order_key" in result.columns
    # check dates parsed
    o1 = result.filter(F.col("order_id") == "O1").collect()[0]
    assert o1["order_date"] == date(2024, 3, 15)
    assert o1["ship_date"] == date(2024, 3, 20)
    # check default filled for ship_mode
    o2 = result.filter(F.col("order_id") == "O2").collect()[0]
    assert o2["ship_mode"] == "unknown"
    assert o2["order_date"] == date(2024, 1, 1)

In [0]:
# Run all tests 

test_functions = [
    test_config_has_struct_schema,
    test_config_has_defaults,
    test_config_has_module_key,
    test_remove_nulls_single_key,
    test_remove_nulls_composite_key,
    test_fill_defaults_products,
    test_fill_defaults_customers,
    test_fill_defaults_orders,
    test_fill_defaults_preserves_existing,
    test_parse_struct_schema_products,
    test_parse_struct_schema_customers,
    test_parse_struct_schema_orders,
    test_apply_schema_drops_extra_columns,
    test_parse_dates_with_format,
    test_parse_dates_null_stays_null,
    test_drop_duplicates,
    test_surrogate_key_added,
    test_dim_transform_products_end_to_end,
    test_dim_transform_customers_end_to_end,
    test_fact_transform_orders_end_to_end,
]

passed, failed = 0, 0
for fn in test_functions:
    try:
        fn()
        passed += 1
        print(f"  [PASSED] {fn.__name__}")
    except Exception as e:
        failed += 1
        print(f"  [FAILED] {fn.__name__}: {e}")

print(f"\nResults: {passed} passed, {failed} failed, {passed + failed} total")